In [87]:
import pandas as pd
import os
import shutil
import json
from copy import deepcopy
import re
from tqdm.notebook import tqdm
import sys

In [88]:
RESULT_DIR = 'results/exp_0428'

result_list = [
                "scn-1_se-24",
                "scn-1_se-25",
                "scn-2_se-24",
                "scn-2_se-25"]

target_sub_dir = [
                    'base','cls','st','cls_st', 'mk']

raw_forder = "eval_embed-bert_inst-whole"
raw_name = "loss.csv"
print_log = False

In [89]:
def convert_flat_json_to_table_format(json_path, save_path=None):
    if save_path is None:
        save_path = json_path

    with open(json_path, "r", encoding="utf-8") as f:
        obj = json.load(f)

    if not isinstance(obj, list):
        raise ValueError("Input JSON must be a list of dicts")

    df = pd.DataFrame(obj)
    table_format = {
        "columns": list(df.columns),
        "data": df.values.tolist()
    }

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(table_format, f, indent=2)


In [90]:
def get_process_config(config, run_id):
    config_dict = deepcopy(config)
    config_dict['run_id'] = run_id

    flatten_keys = ['encoder']

    for key in flatten_keys:
        sub_dict = config_dict[key]
        sub_dict = eval(sub_dict)
        for sub_key, value in sub_dict.items():
            config_dict[f'{key}.{sub_key}'] = value

    remove_keys = ['lr', 'GAMMA', 'is_3d', 'n_eps', 'agents', 'n_bins', 'n_gpus', 'VF_COEF', 'encoder', 'CLIP_EPS', 'ENT_COEF', '_img_dir', '_vid_dir', 'arf_size', 'env_name', 'eval_dir', 'n_agents', 'vrf_size', 'ANNEAL_LR', 'act_shape', 'ckpt_freq', 'eval_freq', 'eval_seed', 'evaluator', 'map_width', 'num_steps', 'overwrite', 'pinpoints', 'wandb_key', 'GAE_LAMBDA', '_numpy_dir', 'activation', 'change_pct', 'initialize', 'multiagent', 'n_freezies', 'NUM_UPDATES', 'empty_start', 'hidden_dims', 'n_eval_envs', 'n_eval_maps', 'profile_fps', 'render_freq', 'ctrl_metrics', 'n_render_eps', 'reward_every', 'wandb_entity', 'wandb_resume', 'MAX_GRAD_NORM', 'eval_map_path', 'update_epochs', 'wandb_project', 'MINIBATCH_SIZE', 'eval_map_types', 'eval_map_width', 'NUM_MINIBATCHES', 'max_board_scans', 'metrics_to_keep', 'static_tile_prob', 'current_iteration', 'gif_frame_duration', 'randomize_map_shape', 'timestep_chunk_size', 'eval_max_board_scans', 'DEBUG', 'eval_randomize_map_shape']

    for key in remove_keys:
        config_dict.pop(key, None)
    # change the values to str
    for key, value in config_dict.items():
        config_dict[key] = str(value)

    return config_dict

In [91]:
for base_folder_name in result_list:
    base_path = os.path.join(RESULT_DIR, base_folder_name)

    for sub_dir in tqdm(os.listdir(base_path), desc="moving in " + base_path, mininterval=0.1):
        if sub_dir not in target_sub_dir:
            continue

        sub_path = os.path.join(base_path, sub_dir)

        for run_dir in os.listdir(sub_path):
            run_path = os.path.join(sub_path, run_dir)
            table_path = os.path.join(run_path, raw_forder)

            if not os.path.exists(table_path):
                if print_log:
                    tqdm.write(f"❌ Failed to move {run_path}: Can't find Folder")
                continue

            ext=raw_name.split(".")[-1]
            initial_raw_path = os.path.join(table_path, raw_name)
            target_raw_path = os.path.join(run_path, "raw.json")

            if not os.path.exists(initial_raw_path):
                if print_log:
                    tqdm.write(f"❌ Failed to move {run_path}: Can't find raw file")
                continue

            if ext == "json":
                try:
                    shutil.move(initial_raw_path, target_raw_path)
                    if not os.listdir(table_path):
                        os.rmdir(table_path)
                    if print_log:
                        tqdm.write(f"Moved: {run_path}")

                except Exception as e:
                    if print_log:
                        tqdm.write(f"❌ Failed to move {run_path}: {e}")

            elif ext == "csv":
                try:
                    # CSV → DataFrame → FlatJSON → JSON
                    df = pd.read_csv(initial_raw_path)
                    df.to_json(target_raw_path, orient="records", indent=2)
                    convert_flat_json_to_table_format(target_raw_path)
                    os.remove(initial_raw_path)
                    if not os.listdir(table_path):
                        os.rmdir(table_path)

                    if print_log:
                        tqdm.write(f"Moved: {run_path}")

                except Exception as e:
                    if print_log:
                        tqdm.write(f"❌ Failed to move {run_path}: {e}")

            else:
                if print_log:
                    tqdm.write(f"❌ Failed to move {run_path}: Invalid file extension")







moving in results/exp_0423\scn-1_se-24:   0%|          | 0/5 [00:00<?, ?it/s]

moving in results/exp_0423\scn-1_se-25:   0%|          | 0/5 [00:00<?, ?it/s]

moving in results/exp_0423\scn-2_se-24:   0%|          | 0/5 [00:00<?, ?it/s]

moving in results/exp_0423\scn-2_se-25:   0%|          | 0/5 [00:00<?, ?it/s]

In [92]:
json_files = ["raw.json"]#, "diversity.json"]

for base_folder_name in result_list:
    base_path = os.path.join(RESULT_DIR, base_folder_name)

    for sub_dir in tqdm(os.listdir(base_path),desc="converting in " +base_path,mininterval=0.1):
        if sub_dir not in target_sub_dir:
            continue

        sub_path = os.path.join(base_path, sub_dir)

        for run_name in os.listdir(sub_path):
            run_path = os.path.join(sub_path, run_name)
            config_path = os.path.join(run_path, "config.json")

            if not os.path.exists(config_path):
                if print_log:
                    tqdm.write(f"⚠️ no config at {config_path}")

                try:
                    instruct_match = re.search(r"inst-([^\s_]+(?:_[^\s_]+)*?)(?:_(enc|tr|model|exp|norm|s)-|$)", run_name)
                    instruct = instruct_match.group(1) if instruct_match else "unknown"
                    seed_match = re.search(r"s-(\d+)(?:_(enc|tr|model|exp|norm|s)-|$)", run_name)
                    seed = int(seed_match.group(1)) if seed_match else -1

                    run_config = {
                        "buffer_ratio": 1,
                        "encoder": str({"output_dim": 64}),
                        "instruct": instruct,
                        "seed": seed
                    }

                    config_dict = get_process_config(run_config, run_name)

                    with open(config_path, "w", encoding="utf-8") as f:
                        json.dump(run_config, f, indent=2)

                    if print_log:
                        tqdm.write(f"✅ Made config at {config_path}")

                except Exception as e:
                    if print_log:
                        tqdm.write(f"❌ Failed to made config at {config_path}: {e}")

            else:
                if print_log:
                    tqdm.write(f"✅ config at {config_path}")
                try:
                    with open(config_path, "r", encoding="utf-8") as f:
                        run_config = json.load(f)

                    config_dict = get_process_config(run_config, run_name)

                    with open(os.path.join(run_path, "config.json"), "w") as f:
                        json.dump(run_config, f, indent=2)

                    if print_log:
                        tqdm.write(f"✅ Update config at {config_path}")

                except Exception as e:
                    if print_log:
                        tqdm.write(f"❌ Failed to update config at {config_path}: {e}")


            for json_name in json_files:
                json_path = os.path.join(run_path, json_name)
                if not os.path.exists(json_path) :
                    if print_log:
                        tqdm.write(f"❌ Failed to convert {json_path}: Can't find original file")
                    continue
                if os.path.getsize(json_path) == 0:
                    if print_log:
                        tqdm.write(f"❌ Failed to convert {json_path}: Original file is empty")
                    continue

                try:
                    with open(json_path, "r", encoding="utf-8") as f:
                        obj = json.load(f)
                    df = pd.DataFrame(obj["data"], columns=obj["columns"])

                    config_df = pd.DataFrame([{f"config.{k}": v for k, v in config_dict.items()}])
                    config_df = pd.concat([config_df] * len(df), ignore_index=True)

                    df = pd.concat([df.reset_index(drop=True), config_df], axis=1)

                    csv_path = os.path.splitext(json_path)[0] + ".csv"
                    df.to_csv(csv_path, index=False)

                    if print_log:
                        tqdm.write(f"✅ Converted to CSV with config: {csv_path}")

                except Exception as e:
                    if print_log:
                        tqdm.write(f"❌ Failed to convert {json_path}: {e}")

converting in results/exp_0423\scn-1_se-24:   0%|          | 0/5 [00:00<?, ?it/s]

converting in results/exp_0423\scn-1_se-25:   0%|          | 0/5 [00:00<?, ?it/s]

converting in results/exp_0423\scn-2_se-24:   0%|          | 0/5 [00:00<?, ?it/s]

converting in results/exp_0423\scn-2_se-25:   0%|          | 0/5 [00:00<?, ?it/s]